<a href="https://colab.research.google.com/github/LucasP01/TFG/blob/main/TFG_Lucas_Paleo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DATASET DE UBER


## Importar dataset y librerias

In [100]:
#Importamos las librerias pandas y numpy para procesar los datos e importamos seaborn para la visualizacion
import pandas as pd
import numpy as np
import seaborn as sns
# from google.colab import files
from sqlalchemy import create_engine


print("Setup Completo")

Setup Completo


In [101]:
#Importamos el archivo desde Github
url = "https://raw.githubusercontent.com/LucasP01/TFG/refs/heads/main/ncr_ride_bookings.csv"

uber = pd.read_csv(url, sep=';')

print("Carga de datos completa")


Carga de datos completa


##Analisis exploratorio de datos

In [102]:
#Visualizamos los primeros 5 registros
uber.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,23/3/2024,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29/11/2024,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,23/8/2024,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,21/10/2024,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,16/9/2024,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [103]:
#Vemos cantidad de filas y columnas
uber.shape

(148767, 21)

In [104]:
#Observamos informacion general del dataset
# Validamos los tipos de datos, la cantidad de nullos y las columnas que tenemos
uber.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148767 entries, 0 to 148766
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               148767 non-null  object 
 1   Time                               148767 non-null  object 
 2   Booking ID                         148767 non-null  object 
 3   Booking Status                     148767 non-null  object 
 4   Customer ID                        148767 non-null  object 
 5   Vehicle Type                       148767 non-null  object 
 6   Pickup Location                    148767 non-null  object 
 7   Drop Location                      148767 non-null  object 
 8   Avg VTAT                           138366 non-null  float64
 9   Avg CTAT                           101175 non-null  float64
 10  Cancelled Rides by Customer        10402 non-null   float64
 11  Reason for cancelling by Customer  1040

In [105]:
#Vemos la distribucion de las columnas numericas
uber.describe()

,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Cancelled Rides by Driver,Incomplete Rides,Booking Value,Ride Distance,Driver Ratings,Customer Rating
count,138366.000000,101175.000000,10402.0,26789.0,8927.0,101175.000000,101175.000000,92248.000000,92248.000000
mean,8.454819,29.150249,1.0,1.0,1.0,508.290230,24.640956,4.230756,4.404301
std,3.773341,8.901703,0.0,0.0,0.0,395.913208,14.002172,0.436741,0.437908
min,2.000000,10.000000,1.0,1.0,1.0,50.000000,1.000000,3.000000,3.000000
25%,5.300000,21.600000,1.0,1.0,1.0,234.000000,12.460000,4.100000,4.200000
50%,8.300000,28.800000,1.0,1.0,1.0,414.000000,23.720000,4.300000,4.500000
75%,11.300000,36.800000,1.0,1.0,1.0,689.000000,36.820000,4.600000,4.800000
max,20.000000,45.000000,1.0,1.0,1.0,4277.000000,50.000000,5.000000,5.000000


## Calidad de datos

In [106]:
#Buscamos registros duplicados
uber.duplicated().sum()

np.int64(0)

In [107]:
#Buscamos datos nulos
uber.isnull().sum()

Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10401
Avg CTAT                              47592
Cancelled Rides by Customer          138365
Reason for cancelling by Customer    138365
Cancelled Rides by Driver            121978
Driver Cancellation Reason           121978
Incomplete Rides                     139840
Incomplete Rides Reason              139840
Booking Value                         47592
Ride Distance                         47592
Driver Ratings                        56519
Customer Rating                       56519
Payment Method                        47592
dtype: int64

In [108]:
#Vemos que Booking ID y Customer ID estan encerrados en comillas.
uber['Booking ID'] = uber['Booking ID'].str.strip('"')
uber['Customer ID'] = uber['Customer ID'].str.strip('"')

print("Comillas limpiadas.")

Comillas limpiadas.


Reglas para imputaciones:

- Numerico continuo (promedio, distancia, calificacion, valor monetario) = rellenamos con la media poara mantener la distribucion general estable
- Numericos discretos (cancelaciones, viajes incompletos) = rellenamos faltantes con 0, si no hay evento es porque no se produjo.

##Observaciones

- Cantidad de registros:
- Columnas:
- Nulos:
- No hay duplicados
- Hay que corregir tipos de datos

##Limpieza de datos

In [109]:
#Visualizamos los datos nulos
uber.isnull().sum()

Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10401
Avg CTAT                              47592
Cancelled Rides by Customer          138365
Reason for cancelling by Customer    138365
Cancelled Rides by Driver            121978
Driver Cancellation Reason           121978
Incomplete Rides                     139840
Incomplete Rides Reason              139840
Booking Value                         47592
Ride Distance                         47592
Driver Ratings                        56519
Customer Rating                       56519
Payment Method                        47592
dtype: int64

Lo que hacemos con los nulos es lo siguiente:
- Avg VTAT usamos la media de CTAT
- Avg CTAT, Booking Value, Ride Distance y Rating con sus respectivas medias
- Las Cancelattion counts con 0
- Columnas categoricas con "none"
- Payment Method con su moda

In [110]:
uber['Avg VTAT'] = uber["Avg VTAT"].fillna(uber["Avg CTAT"].mean())
uber['Avg CTAT'] = uber["Avg CTAT"].fillna(uber["Avg CTAT"].mean())
uber['Cancelled Rides by Customer'] = uber['Cancelled Rides by Customer'].fillna(0)
uber['Reason for cancelling by Customer'] = uber['Reason for cancelling by Customer'].fillna("none")
uber['Cancelled Rides by Driver'] = uber['Cancelled Rides by Driver'].fillna(0)
uber['Driver Cancellation Reason'] = uber['Driver Cancellation Reason'].fillna("none")
uber['Incomplete Rides'] = uber['Incomplete Rides'].fillna(0)
uber['Incomplete Rides Reason'] = uber['Incomplete Rides Reason'].fillna("none")
uber['Booking Value'] = uber['Booking Value'].fillna(uber['Booking Value'].mean())
uber['Ride Distance'] = uber['Ride Distance'].fillna(uber['Ride Distance'].mean())
uber['Driver Ratings'] = uber['Driver Ratings'].fillna(uber['Driver Ratings'].mean())
uber['Customer Rating'] = uber['Customer Rating'].fillna(uber['Customer Rating'].mean())
uber['Payment Method'].fillna(uber['Payment Method'].mode()[0], inplace=True)

print("Imputacion de datos completa")


Imputacion de datos completa


C:\Users\lucas\AppData\Local\Temp\ipykernel_21484\3928074597.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  uber['Payment Method'].fillna(uber['Payment Method'].mode()[0], inplace=True)


In [111]:
#Revisamos que esten OK los reemplazados
uber.isnull().sum()

Date                                 0
Time                                 0
Booking ID                           0
Booking Status                       0
Customer ID                          0
Vehicle Type                         0
Pickup Location                      0
Drop Location                        0
Avg VTAT                             0
Avg CTAT                             0
Cancelled Rides by Customer          0
Reason for cancelling by Customer    0
Cancelled Rides by Driver            0
Driver Cancellation Reason           0
Incomplete Rides                     0
Incomplete Rides Reason              0
Booking Value                        0
Ride Distance                        0
Driver Ratings                       0
Customer Rating                      0
Payment Method                       0
dtype: int64

Para los datos asignamos los siguientes datatypes:
- Numericos : INT o Float (depende si se requieren decimales o no)
- Categorias: Category
- Fechas: datetime
- Texto: string

In [112]:
#Procedemos con los cambios del tipo de dato
uber['Date'] = pd.to_datetime(uber['Date'], errors='coerce')
uber['Time'] = pd.to_datetime(uber['Time'], format = '%H:%M:%S', errors='coerce').dt.time
uber['Booking ID'] = uber['Booking ID']. astype('string')
uber['Booking Status'] = uber['Booking Status'].astype('string')
uber['Customer ID'] = uber['Customer ID']. astype('string')
uber['Vehicle Type'] = uber['Vehicle Type']. astype('string')
uber['Pickup Location'] = uber['Pickup Location']. astype('string')
uber['Drop Location'] = uber['Drop Location']. astype('string')
uber['Cancelled Rides by Customer'] = uber['Cancelled Rides by Customer']. astype('int')
uber['Reason for cancelling by Customer'] = uber['Reason for cancelling by Customer']. astype('string')
uber['Cancelled Rides by Driver'] = uber['Cancelled Rides by Driver']. astype('int')
uber['Driver Cancellation Reason'] = uber['Driver Cancellation Reason']. astype('string')
uber['Incomplete Rides'] = uber['Incomplete Rides']. astype('int')
uber['Incomplete Rides Reason'] = uber['Incomplete Rides Reason']. astype('string')
uber['Payment Method'] = uber['Payment Method']. astype('string')

print("Cambio de datatype completo")



C:\Users\lucas\AppData\Local\Temp\ipykernel_21484\3020124969.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  uber['Date'] = pd.to_datetime(uber['Date'], errors='coerce')


Cambio de datatype completo


In [113]:
#Revisamos los tipos de datos
uber.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148767 entries, 0 to 148766
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Date                               148767 non-null  datetime64[ns]
 1   Time                               148767 non-null  object        
 2   Booking ID                         148767 non-null  string        
 3   Booking Status                     148767 non-null  string        
 4   Customer ID                        148767 non-null  string        
 5   Vehicle Type                       148767 non-null  string        
 6   Pickup Location                    148767 non-null  string        
 7   Drop Location                      148767 non-null  string        
 8   Avg VTAT                           148767 non-null  float64       
 9   Avg CTAT                           148767 non-null  float64       
 10  Cancelled Rides by C

##Carga en SQL

##Exportamos el CSV

In [114]:
# En caso de no poder conectarse a la base de  datos, dejo esto para que descarguen el CSV y se conecte al PowerBI
# uber.to_csv('uber.csv', index=False)
# files.download('uber.csv')

# print("✅ Archivo CSV exportado: dataset_limpio.csv")

##Insertamos los datos en la BBDD


In [115]:
import pymysql
import warnings

# Configuramos la conexion
USER = 'root'
PASSWORD = '1234'
HOST = 'localhost'
PORT = 3306 
DATABASE = 'uber_db'

# Query para crear la tabla
query_crear_tabla = """
CREATE TABLE IF NOT EXISTS Uber_Rides (
    `Date` DATETIME NOT NULL,
    `Time` TIME NOT NULL,
    `Booking_ID` VARCHAR(20) NOT NULL,
    `Booking_Status` VARCHAR(30) NOT NULL,
    `Customer_ID` VARCHAR(20) NOT NULL,
    `Vehicle_Type` VARCHAR(20) NOT NULL,
    `Pickup_Location` VARCHAR(100) NOT NULL,
    `Drop_Location` VARCHAR(100) NOT NULL,
    `Avg_VTAT` DOUBLE NULL,
    `Avg_CTAT` DOUBLE NULL,
    `Cancelled_Rides_by_Customer` INT NOT NULL,
    `Reason_for_cancelling_by_Customer` VARCHAR(255) NULL,
    `Cancelled_Rides_by_Driver` INT NOT NULL,
    `Driver_Cancellation_Reason` VARCHAR(255) NULL,
    `Incomplete_Rides` INT NULL,
    `Incomplete_Rides_Reason` VARCHAR(255) NULL,
    `Booking_Value` DECIMAL(10, 2) NULL,
    `Ride_Distance` DECIMAL(10, 2) NULL,
    `Driver_Ratings` DECIMAL(3, 1) NULL,
    `Customer_Rating` DECIMAL(3, 1) NULL,
    `Payment_Method` VARCHAR(20) NULL,
    PRIMARY KEY (`Booking_ID`)
)
"""

try:
    # Nos conectamos al servidor, en este caso Local
    conn = pymysql.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        port=PORT
    )
    conn.autocommit = True  # Para que los 'CREATE' se ejecuten automáticamente
    cursor = conn.cursor()
    
    # Creamos la Base de Datos
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
    print(f"Base de datos '{DATABASE}' asegurada.")
    
    # Seleccionamos la base de datos
    cursor.execute(f"USE {DATABASE}")
    
    # Creamos la Tabla (ignorando warnings si ya existe)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cursor.execute(query_crear_tabla)
    
    print(f"Tabla 'Uber_Rides' asegurada.")
    print("\nEstructura de BBDD lista")

except pymysql.Error as e:
    print(f"Error al conectar o crear la BBDD/Tabla: {e}")
finally:
    # Cerramos la conexión inicial
    if 'cursor' in locals():
        cursor.close()
    if 'conn' in locals():
        conn.close()

Base de datos 'uber_db' asegurada.
Tabla 'Uber_Rides' asegurada.

Estructura de BBDD lista


In [116]:
from sqlalchemy import create_engine

# (Pandas/SQLAlchemy necesita este formato de string de conexión)
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

# Mapeo de columnas
column_map = {
    'Date': 'Date', 'Time': 'Time', 'Booking ID': 'Booking_ID', 
    'Booking Status': 'Booking_Status', 'Customer ID': 'Customer_ID', 
    'Vehicle Type': 'Vehicle_Type', 'Pickup Location': 'Pickup_Location', 
    'Drop Location': 'Drop_Location', 'Avg VTAT': 'Avg_VTAT', 
    'Avg CTAT': 'Avg_CTAT', 'Cancelled Rides by Customer': 'Cancelled_Rides_by_Customer', 
    'Reason for cancelling by Customer': 'Reason_for_cancelling_by_Customer', 
    'Cancelled Rides by Driver': 'Cancelled_Rides_by_Driver', 
    'Driver Cancellation Reason': 'Driver_Cancellation_Reason', 
    'Incomplete Rides': 'Incomplete_Rides', 'Incomplete Rides Reason': 'Incomplete_Rides_Reason', 
    'Booking Value': 'Booking_Value', 'Ride Distance': 'Ride_Distance', 
    'Driver Ratings': 'Driver_Ratings', 'Customer Rating': 'Customer_Rating', 
    'Payment Method': 'Payment_Method'
}

# Conexion con SQLAlchemy e insercion de datos
try:
    #Renombramos columnas del DataFrame
    uber.rename(columns=column_map, inplace=True)
    print("Columnas del DataFrame renombradas.")
    
    engine = create_engine(connection_string)
    
    #Iniciamos la carga de datos
    print(f"Iniciando carga de {len(uber)} filas en 'Uber_Rides'...")
    uber.to_sql(
        name='Uber_Rides',     # Nombre de la tabla en MySQL
        con=engine,           # El motor de conexión
        if_exists='append',   # 'append' = agrega los datos.
        index=False,          # No guardamos el índice de Pandas
        chunksize=10000       # Carga en lotes de 10k filas (óptimo)
    )

    print("\nÉXITO TOTAL")
    print(f"Se cargaron {len(uber)} filas correctamente en 'uber_db.Uber_Rides'.")

except NameError:
    print("Error: El DataFrame 'uber' no está definido. Asegurate de correr la celda de procesamiento (Celda 3) primero.")
except Exception as e:
    print(f"Ocurrió un error durante la carga de datos a SQL: {e}")
finally:
    # Cerramos la conexión del "motor"
    if 'engine' in locals():
        engine.dispose()
        print("Conexión de SQLAlchemy (engine) cerrada.")

Columnas del DataFrame renombradas.
Iniciando carga de 148767 filas en 'Uber_Rides'...


C:\Users\lucas\AppData\Local\Temp\ipykernel_21484\3395585643.py:32: UserWarning: The provided table name 'Uber_Rides' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  uber.to_sql(



ÉXITO TOTAL
Se cargaron 148767 filas correctamente en 'uber_db.Uber_Rides'.
Conexión de SQLAlchemy (engine) cerrada.
